# Double descent — book graphics 8: polynomial double descent table (bias / variance vs degree)

Bias–variance decomposition of the minimum-norm (pseudoinverse) Legendre fit as a function of polynomial degree, with the **same data setup for every degree**: `N_TRAIN` training points per dataset, `N_DATASETS` datasets (seeds 0…N−1), noise σ = 0.2 on $f(x) = \tfrac12 x^2$. The interpolation threshold is at degree `N_TRAIN − 1`.

For each degree the table reports, averaged over $x \in [-2, 2]$:

- **bias²** = mean over x of $(f(x) - \bar f(x))^2$, where $\bar f$ is the mean fit across datasets
- **variance** = mean over x and datasets of $(\hat f_D(x) - \bar f(x))^2$
- **σ²** = 0.04 and **total** = bias² + variance + σ² (= the expected test MSE)
- **median per-dataset variance** and **worst-3 share** — how much of the variance comes from the three worst datasets (a heavy-tail diagnostic)
- **mean / median test MSE** measured directly on each dataset's held-out test points

`X_SAMPLING = 'uniform'` draws training x uniformly (as in the video); near-collinear training points then produce catastrophic interpolants near the threshold, and the *means* are dominated by a few datasets. `'grid'` places training x on a jittered grid (no near-collinear points) and gives a clean textbook double-descent curve.


## 1. Config

In [ ]:
from pathlib import Path
import numpy as np
import scipy.special
import matplotlib.pyplot as plt
import matplotlib as mpl

CHILL_BROWN = '#948979'; LT_BLUE = '#5badb6'; ORANGE = '#eb8423'; GREEN = '#419c52'
BLUE = '#236c94'; RED = '#d73b2f'; PINK = '#ed5e78'; YELLOW = '#fcc947'
AXIS_COLOR = CHILL_BROWN
BIAS_COLOR, VAR_COLOR, NOISE_COLOR, TOTAL_COLOR = PINK, BLUE, GREEN, CHILL_BROWN
MEAN_MSE_COLOR, MEDIAN_MSE_COLOR = ORANGE, RED

# ── Setup ──────────────────────────────────────────────────────────────────
NOISE_LEVEL = 0.2
X_MIN, X_MAX = -2.0, 2.0
def f(x): return 0.5 * x**2
N_TRAIN     = 15                     # training points per dataset  (interpolation threshold = degree N_TRAIN - 1)
N_TEST      = 15                     # held-out test points per dataset
N_DATASETS  = 256                    # seeds 0 .. N_DATASETS-1
DEGREES     = list(range(1, 31))     # e.g. list(range(1, 31)) + [40, 50, 60, 80, 100] to see the second descent run on
X_SAMPLING  = 'uniform'              # 'uniform' (as in the video) or 'grid' (jittered grid: no near-collinear training points)
GRID_JITTER = 0.25                   # for 'grid': jitter as a fraction of the grid spacing
N_EVAL_X    = 128                    # x grid for the bias / variance integrals

# ── Plot ───────────────────────────────────────────────────────────────────
AX_WIDTH_IN, AX_HEIGHT_IN, MARGIN_IN = 5.0, 3.4, 0.6
LINEWIDTH, DOT_SIZE, FONTSIZE, AXIS_LINEWIDTH = 1.5, 18, 10, 1.0
LOG_Y        = True
SHOW_THRESHOLD = True                # dashed vertical line at the interpolation threshold
TRANSPARENT  = True

USE_TEX = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['cmr10', 'Computer Modern Roman', 'DejaVu Serif']
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['axes.formatter.use_mathtext'] = True
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['text.usetex'] = USE_TEX
plt.rcParams['svg.fonttype'] = 'none'

OUT_DIR = Path('/Users/stephen/Library/CloudStorage/Dropbox-Stephencwelch/welch_labs/ai_book_vol_2/5_double_descent/graphics/poly_double_descent_table')
SAVE    = True
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. Helpers

In [ ]:
def legendre_features(x, degree, x_min=X_MIN, x_max=X_MAX):
    x_scaled = 2 * (x - x_min) / (x_max - x_min) - 1
    return scipy.special.eval_legendre(np.arange(degree + 1)[:, None], x_scaled).T

def fit_legendre_pinv(x_train, y_train, degree):
    return np.linalg.pinv(legendre_features(x_train, degree)) @ y_train

def eval_legendre_poly(beta, x, degree):
    return legendre_features(x, degree) @ beta

def make_dataset(seed):
    rng = np.random.RandomState(seed)
    if X_SAMPLING == 'grid':
        spacing = (X_MAX - X_MIN) / (N_TRAIN - 1)
        x_tr = np.linspace(X_MIN, X_MAX, N_TRAIN) + rng.uniform(-GRID_JITTER, GRID_JITTER, N_TRAIN) * spacing
        x_tr = np.clip(x_tr, X_MIN, X_MAX)
        x_te = rng.uniform(X_MIN, X_MAX, N_TEST)
    else:
        x = rng.uniform(X_MIN, X_MAX, N_TRAIN + N_TEST)
        x_tr, x_te = x[:N_TRAIN], x[N_TRAIN:]
    y_tr = f(x_tr) + NOISE_LEVEL * rng.randn(N_TRAIN)
    y_te = f(x_te) + NOISE_LEVEL * rng.randn(N_TEST)
    return x_tr, y_tr, x_te, y_te


## 3. Compute the table

In [ ]:
all_x = np.linspace(X_MIN, X_MAX, N_EVAL_X)
all_y = f(all_x)
sigma2 = NOISE_LEVEL**2
datasets = [make_dataset(s) for s in range(N_DATASETS)]

rows = []
fits_by_degree = {}
for d in DEGREES:
    fits, test_mse = [], []
    for x_tr, y_tr, x_te, y_te in datasets:
        beta = fit_legendre_pinv(x_tr, y_tr, d)
        fits.append(eval_legendre_poly(beta, all_x, d))
        test_mse.append(np.mean((y_te - eval_legendre_poly(beta, x_te, d))**2))
    fits = np.array(fits); test_mse = np.array(test_mse)
    mean_fit = fits.mean(0)
    per_ds_var = np.mean((fits - mean_fit)**2, axis=1)          # each dataset's contribution to the variance
    bias2 = np.mean((all_y - mean_fit)**2)
    var   = per_ds_var.mean()
    worst3 = np.sort(per_ds_var)[::-1][:3].sum() / per_ds_var.sum()
    rows.append(dict(degree=d, bias2=bias2, variance=var, sigma2=sigma2, total=bias2 + var + sigma2,
                     median_ds_var=np.median(per_ds_var), worst3_share=worst3,
                     mean_test_mse=test_mse.mean(), median_test_mse=np.median(test_mse)))
    fits_by_degree[d] = fits

hdr = f"{'deg':>4} | {'bias^2':>11} {'variance':>11} {'sigma^2':>8} {'total':>11} | {'median ds var':>13} {'worst-3':>7} | {'mean test MSE':>13} {'median test MSE':>15}"
print(f'{N_TRAIN} training points, {N_DATASETS} datasets, x sampling = {X_SAMPLING}, interpolation threshold = degree {N_TRAIN - 1}')
print(hdr); print('-' * len(hdr))
for r in rows:
    print(f"{r['degree']:>4} | {r['bias2']:>11.4g} {r['variance']:>11.4g} {r['sigma2']:>8.3f} {r['total']:>11.4g} | "
          f"{r['median_ds_var']:>13.4g} {r['worst3_share']:>7.1%} | {r['mean_test_mse']:>13.4g} {r['median_test_mse']:>15.4g}")

# CSV for the book / caption
import csv
with open(OUT_DIR / f'bias_variance_table_ntrain{N_TRAIN}_{X_SAMPLING}.csv', 'w', newline='') as fh:
    w = csv.DictWriter(fh, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)


## 4. Plot

In [ ]:
def exact_axes(width_in, height_in, margin=MARGIN_IN):
    fig = plt.figure(figsize=(width_in + 2 * margin, height_in + 2 * margin))
    ax = fig.add_axes([margin / fig.get_figwidth(), margin / fig.get_figheight(),
                       width_in / fig.get_figwidth(), height_in / fig.get_figheight()])
    return fig, ax

def style_ax(ax, color=AXIS_COLOR):
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[:].set_color(color); ax.spines[:].set_linewidth(AXIS_LINEWIDTH)
    ax.tick_params(colors=color, labelsize=FONTSIZE, width=AXIS_LINEWIDTH, which='both')

degs = np.array([r['degree'] for r in rows])
def col(k): return np.array([r[k] for r in rows])

def draw_decomposition(ax):
    """bias^2, variance, sigma^2 and their sum vs degree."""
    for k, c in (('bias2', BIAS_COLOR), ('variance', VAR_COLOR), ('sigma2', NOISE_COLOR), ('total', TOTAL_COLOR)):
        ax.plot(degs, col(k), color=c, lw=LINEWIDTH, zorder=2)
        ax.scatter(degs, col(k), color=c, s=DOT_SIZE, zorder=3, linewidths=0)
    finish(ax)

def draw_test_mse(ax):
    """expected test MSE (mean over datasets) and the median dataset's test MSE vs degree."""
    for k, c in (('mean_test_mse', MEAN_MSE_COLOR), ('median_test_mse', MEDIAN_MSE_COLOR)):
        ax.plot(degs, col(k), color=c, lw=LINEWIDTH, zorder=2)
        ax.scatter(degs, col(k), color=c, s=DOT_SIZE, zorder=3, linewidths=0)
    finish(ax)

def finish(ax):
    if LOG_Y: ax.set_yscale('log')
    if SHOW_THRESHOLD:
        ax.axvline(N_TRAIN - 1, color=AXIS_COLOR, lw=AXIS_LINEWIDTH, ls=(0, (4, 3)), zorder=1)
    ax.set_xlim(degs.min() - 0.5, degs.max() + 0.5)
    style_ax(ax)

fig, ax = exact_axes(AX_WIDTH_IN, AX_HEIGHT_IN); draw_decomposition(ax)
ax.set_title(f'bias$^2$ (pink), variance (blue), $\\sigma^2$ (green), total (brown) - {N_TRAIN} train pts, {X_SAMPLING}', color=AXIS_COLOR, fontsize=FONTSIZE); plt.show()
fig, ax = exact_axes(AX_WIDTH_IN, AX_HEIGHT_IN); draw_test_mse(ax)
ax.set_title('test MSE: mean over datasets (orange), median dataset (red)', color=AXIS_COLOR, fontsize=FONTSIZE); plt.show()


## 5. Export

In [ ]:
def save(fig, stem):
    for ext in ('svg', 'png'):
        fig.savefig(OUT_DIR / f'{stem}.{ext}', dpi=300, transparent=TRANSPARENT, bbox_inches='tight')
    plt.close(fig)

if SAVE:
    tag = f'ntrain{N_TRAIN}_{X_SAMPLING}'
    fig, ax = exact_axes(AX_WIDTH_IN, AX_HEIGHT_IN); draw_decomposition(ax); save(fig, f'decomposition_vs_degree_{tag}')
    fig, ax = exact_axes(AX_WIDTH_IN, AX_HEIGHT_IN); draw_test_mse(ax);      save(fig, f'test_mse_vs_degree_{tag}')
    print('saved to', OUT_DIR)
    for p in sorted(OUT_DIR.iterdir()):
        print('  ', p.name)
